In [1]:
import os
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# 1. Load Waqas's cleaned dataset
df = pd.read_csv("../Data/cleaned_cbc.csv")

# 2. Define exact feature columns and target
FEATURE_COLS = ["Age", "Gender", "RBC", "PCV", "MCV", "MCH", "MCHC", "RDW", "WBC", "PLT", "HGB"]
X = df[FEATURE_COLS]
y = df["Leukopenia_Risk"]

# 3. Stratified Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Fit StandardScaler strictly on training data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Model 1: Full CBC Benchmark (Production Candidates)
models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5, weights="distance"),
    "SVM": SVC(probability=True, class_weight="balanced", random_state=42)
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1-Score": round(f1_score(y_test, y_pred, zero_division=0), 4)
    })

results_df = pd.DataFrame(results).sort_values(by=["Recall", "F1-Score", "Accuracy"], ascending=False)
print("--- Model 1: Full CBC Benchmark ---")
print(results_df.to_string(index=False))

# 6. Export Winning Model (Random Forest) & Scaler for Production
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

os.makedirs("../models", exist_ok=True)
joblib.dump(best_model, "../models/best_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(FEATURE_COLS, "../models/feature_cols.pkl")
print(f"\nExported Production Model: {best_model_name}")

# 7. Experiment 2: Secondary Markers Only (Without WBC — Target Leakage Check)
X_no_wbc = df[["Age", "Gender", "RBC", "PCV", "MCV", "MCH", "MCHC", "RDW", "PLT", "HGB"]]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_no_wbc, y, test_size=0.2, random_state=42, stratify=y)
sc2 = StandardScaler()
rf_secondary = RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42)
rf_secondary.fit(sc2.fit_transform(X_tr2), y_tr2)
preds2 = rf_secondary.predict(sc2.transform(X_te2))
print(f"\n--- Experiment 2 (Without WBC) ---")
print(f"Random Forest (No WBC) -> Accuracy: {accuracy_score(y_te2, preds2):.4f} | Recall: {recall_score(y_te2, preds2):.4f}")

C:\Users\Farman Ali\AppData\Roaming\Python\Python314\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


--- Model 1: Full CBC Benchmark ---
              Model  Accuracy  Precision  Recall  F1-Score
      Random Forest    1.0000     1.0000  1.0000      1.00
      Decision Tree    1.0000     1.0000  1.0000      1.00
                SVM    0.9722     0.6000  1.0000      0.75
Logistic Regression    0.9444     0.4286  1.0000      0.60
                KNN    0.9722     1.0000  0.3333      0.50

Exported Production Model: Random Forest

--- Experiment 2 (Without WBC) ---
Random Forest (No WBC) -> Accuracy: 0.9722 | Recall: 0.3333
